# UPSHADOW 上影线因子构造（预热集：2004–2005；训练集：2006–2017）

本 Notebook 只负责因子生产：

1. 从 `Data/stocks_daily_04-25.parquet` 读取 2004 年起的行情；
2. 使用 2004–2005 年数据为 20 日滚动窗口预热；
3. 只在 2006-01-04 至 2017-12-29 的正式训练区间做横截面处理和因子输出；
4. 每日进行 MAD 去极值与 Z-score 标准化；
5. 保存标准因子 Parquet，交给 `Back_test/通用因子回测模板.ipynb`。

> 预热数据只参与历史滚动，不进入训练因子文件，也不参与训练期的覆盖率统计。本文件不构造未来收益、不计算 IC、不做分组回测。


## 1. 导入依赖与参数

默认参数固定为预热期和正式训练期。环境变量仅用于快速冒烟测试，不影响 Notebook 中的默认值。


In [ ]:
from pathlib import Path
from IPython.display import display
import os

os.environ.setdefault('MPLCONFIGDIR', '/tmp/upshadow_matplotlib_cache')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd())
DATA_PATH = PROJECT_DIR / 'Data' / 'stocks_daily_04-25.parquet'
NOTEBOOK_DIR = PROJECT_DIR / 'Factor'

# 2004–2005 只用于滚动窗口预热；正式因子仍严格输出 2006–2017。
WARMUP_START_DATE = os.getenv('UPSHADOW_WARMUP_START_DATE', '2004-01-01')
START_DATE = os.getenv('UPSHADOW_START_DATE', '2006-01-04')
END_DATE = os.getenv('UPSHADOW_END_DATE', '2017-12-29')
OUTPUT_DIR = Path(os.getenv(
    'UPSHADOW_OUTPUT_DIR',
    str(NOTEBOOK_DIR / 'upshadow_output_20060104_20171229')
))

ROLLING_WINDOW = 20
MIN_PERIODS = ROLLING_WINDOW  # 只输出完整 20 个有效观测的滚动均值
MAD_SCALE = 3.0
EXCLUDE_ST = True

WARMUP_START_TS = pd.Timestamp(WARMUP_START_DATE)
START_TS = pd.Timestamp(START_DATE)
END_TS = pd.Timestamp(END_DATE)
WARMUP_START_INT = int(WARMUP_START_TS.strftime('%Y%m%d'))
START_INT = int(START_TS.strftime('%Y%m%d'))
END_INT = int(END_TS.strftime('%Y%m%d'))

assert WARMUP_START_TS < START_TS <= END_TS
assert DATA_PATH.exists(), f'找不到行情文件：{DATA_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('行情文件：', DATA_PATH)
print('预热区间：', WARMUP_START_DATE, '至', (START_TS - pd.Timedelta(days=1)).date())
print('训练区间：', START_DATE, '至', END_DATE)
print('输出目录：', OUTPUT_DIR)
print('pandas / pyarrow：', pd.__version__, '/', pa.__version__)


## 2. 分批读取 Parquet

只读取计算所需字段，并在 Arrow 层过滤日期。读取范围从预热期起点开始，但不会载入训练期结束后的验证集和测试集数据。


In [ ]:
RAW_COLUMNS = [
    '股票代码', '股票名称', '交易日',
    '开盘价', '最高价', '最低价', '收盘价', '复权因子'
]


def load_market_window(path, start_int, end_int, batch_size=300_000):
    parquet_file = pq.ParquetFile(path)
    selected = []
    scanned = 0

    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=RAW_COLUMNS,
        use_threads=True,
    ):
        table = pa.Table.from_batches([batch])
        date_column = table['交易日']
        mask = pc.and_(
            pc.greater_equal(date_column, pa.scalar(start_int)),
            pc.less_equal(date_column, pa.scalar(end_int)),
        )
        filtered = table.filter(mask)
        if filtered.num_rows:
            selected.append(filtered)
        scanned += table.num_rows

    assert selected, '预热及训练区间内没有读取到数据'
    result = pa.concat_tables(selected).to_pandas()
    print(f'扫描 {scanned:,} 行，预热及训练区间保留 {len(result):,} 行')
    return result


market = load_market_window(DATA_PATH, WARMUP_START_INT, END_INT)
market = market.rename(columns={
    '股票代码': 'code',
    '股票名称': 'name',
    '交易日': 'date_int',
    '开盘价': 'open',
    '最高价': 'high',
    '最低价': 'low',
    '收盘价': 'close',
    '复权因子': 'adj_factor',
})
market['date'] = pd.to_datetime(market['date_int'], format='%Y%m%d')
market = market.drop(columns='date_int')

# 去掉上市前等整行无行情记录；预热期与训练期采用完全相同的数据质量规则。
market = market.dropna(subset=['open', 'high', 'low', 'close'], how='all')
market = market.sort_values(['code', 'date']).reset_index(drop=True)

assert market['date'].min() >= WARMUP_START_TS
assert market['date'].max() <= END_TS
assert (market['date'] < START_TS).any(), '没有读到正式训练期之前的预热数据'
assert not market.duplicated(['date', 'code']).any(), '存在重复的 股票-交易日 记录'

print('实际读取日期：', market['date'].min().date(), '至', market['date'].max().date())
print('预热行情行数：', f"{(market['date'] < START_TS).sum():,}")
print('训练行情行数：', f"{market['date'].between(START_TS, END_TS).sum():,}")
print('股票数量：', f"{market['code'].nunique():,}")
display(market.head())


## 3. 数据质量与可投资样本

OHLC 关系异常、零振幅、代码缺失的记录不参与因子计算。若 `EXCLUDE_ST=True`，名称中带 ST 的股票不进入当日横截面。质量检查覆盖预热期和训练期。


In [ ]:
for column in ['open', 'high', 'low', 'close', 'adj_factor']:
    market[column] = pd.to_numeric(market[column], errors='coerce')

valid_ohlc = (
    market['code'].notna()
    & (market[['open', 'high', 'low', 'close']] > 0).all(axis=1)
    & (market['high'] >= market[['open', 'close', 'low']].max(axis=1))
    & (market['low'] <= market[['open', 'close', 'high']].min(axis=1))
)

if EXCLUDE_ST:
    is_st = market['name'].fillna('').str.upper().str.contains('ST', regex=False)
else:
    is_st = pd.Series(False, index=market.index)

quality_summary = pd.Series({
    '行情行数': len(market),
    'OHLC有效行数': int(valid_ohlc.sum()),
    'OHLC异常或缺失行数': int((~valid_ohlc).sum()),
    'ST记录数': int(is_st.sum()),
    '重复股票日期数': int(market.duplicated(['date', 'code']).sum()),
})
display(quality_summary.to_frame('数量'))

## 4. 计算 UPSHADOW 因子

单日定义：

```text
实体顶部 = max(开盘价, 收盘价)
单日上影线比例 = (最高价 - 实体顶部) / (最高价 - 最低价)
UPSHADOW_RAW = 单日上影线比例最近 20 个有效观测的均值
```

滚动计算只使用当前及历史记录，并要求完整的 20 个有效观测。2004–2005 年行情先参与滚动预热，完成计算后才截取 2006–2017 年正式训练区间，因此训练首日不会因为数据集边界导致冷启动缺失。新上市、长期停牌或 OHLC 无效造成的个股历史不足仍会保留为空，避免用不完整窗口制造因子值。


In [ ]:
body_top = market[['open', 'close']].max(axis=1)
daily_range = market['high'] - market['low']

market['upshadow_daily'] = (
    (market['high'] - body_top) / daily_range.where(daily_range > 0)
)
market.loc[
    ~valid_ohlc | ~market['upshadow_daily'].between(0, 1),
    'upshadow_daily'
] = np.nan

code_group = market.groupby('code', sort=False)['upshadow_daily']
market['rolling_valid_obs'] = code_group.transform(
    lambda values: values.rolling(ROLLING_WINDOW, min_periods=1).count()
)
market['upshadow_raw'] = code_group.transform(
    lambda values: values.rolling(
        ROLLING_WINDOW,
        min_periods=MIN_PERIODS,
    ).mean()
)

# ST 仅在横截面处理前排除；其历史行情仍可用于股票恢复正常名称后的滚动窗口。
market.loc[is_st, 'upshadow_raw'] = np.nan

# 验证正式训练首日：所有具备完整窗口且可投资的记录都必须已有因子值。
target_mask = market['date'].between(START_TS, END_TS)
assert target_mask.any(), '正式训练区间内没有行情数据'
first_training_date = market.loc[target_mask, 'date'].min()
first_day_mask = market['date'].eq(first_training_date)
first_day_eligible = (
    first_day_mask
    & ~is_st
    & market['rolling_valid_obs'].ge(ROLLING_WINDOW)
)
assert market.loc[first_day_eligible, 'upshadow_raw'].notna().all(), (
    '训练首日仍存在由滚动计算造成的意外缺失'
)

warmup_diagnostics = pd.Series({
    '正式训练首日': first_training_date,
    '首日行情记录数': int(first_day_mask.sum()),
    '首日完整20期且可投资记录数': int(first_day_eligible.sum()),
    '首日有效原始因子数': int(
        market.loc[first_day_mask, 'upshadow_raw'].notna().sum()
    ),
    '首日因历史不足而缺失数': int(
        (first_day_mask & market['rolling_valid_obs'].lt(ROLLING_WINDOW)).sum()
    ),
})
display(warmup_diagnostics.to_frame('值'))

# 从这里开始只保留正式训练期；预热记录不参与横截面标准化和最终输出。
market = market.loc[target_mask].copy().reset_index(drop=True)
assert market['date'].between(START_TS, END_TS).all()

display(
    market.loc[market['upshadow_raw'].notna(),
               ['date', 'code', 'upshadow_daily', 'rolling_valid_obs', 'upshadow_raw']].head()
)


## 5. 每日横截面去极值与标准化

只对正式训练区间逐日进行 MAD 去极值和 Z-score。若某日 MAD 为 0，则该日跳过去极值但仍执行标准化。


In [ ]:
date_group = market.groupby('date', sort=False)['upshadow_raw']
daily_median = date_group.transform('median')
absolute_deviation = (market['upshadow_raw'] - daily_median).abs()
daily_mad = absolute_deviation.groupby(market['date'], sort=False).transform('median')

lower_limit = daily_median - MAD_SCALE * daily_mad
upper_limit = daily_median + MAD_SCALE * daily_mad
clipped = market['upshadow_raw'].clip(lower=lower_limit, upper=upper_limit)
market['upshadow_winsor'] = clipped.where(daily_mad > 0, market['upshadow_raw'])

winsor_group = market.groupby('date', sort=False)['upshadow_winsor']
daily_mean = winsor_group.transform('mean')
daily_std = winsor_group.transform('std')
market['upshadow_factor'] = (
    (market['upshadow_winsor'] - daily_mean) / daily_std.where(daily_std > 0)
)

factor_coverage = market.groupby('date').agg(
    market_rows=('code', 'size'),
    valid_factor=('upshadow_factor', 'count'),
)
factor_coverage['coverage'] = (
    factor_coverage['valid_factor'] / factor_coverage['market_rows']
)

display(factor_coverage.describe().T)

## 6. 保存标准因子文件

这是因子生产与通用回测的分界。输出仅包含正式训练期，包含原始因子、去极值因子和最终标准化因子；`asof_date` 明确表示每条因子只使用截至信号日的信息。

通用回测只会读取 `date`、`code` 和配置指定的因子列，不会载入其他宽表字段。


In [ ]:
factor_columns = [
    'date', 'code', 'name', 'upshadow_daily',
    'upshadow_raw', 'upshadow_winsor', 'upshadow_factor'
]
factor_panel = market.loc[
    market['upshadow_factor'].notna(), factor_columns
].copy()
factor_panel['asof_date'] = factor_panel['date']

assert factor_panel['date'].between(START_TS, END_TS).all()
assert factor_panel['date'].min() == market['date'].min()
assert not factor_panel.duplicated(['date', 'code']).any()

date_tag = f"{START_TS:%Y%m%d}_{END_TS:%Y%m%d}"
factor_path = OUTPUT_DIR / f'upshadow_factor_{date_tag}.parquet'
factor_panel.to_parquet(factor_path, index=False, compression='zstd')
factor_coverage.to_csv(OUTPUT_DIR / 'factor_coverage.csv', index_label='date')

factor_summary = pd.Series({
    '因子记录数': len(factor_panel),
    '股票数量': factor_panel['code'].nunique(),
    '有效因子首日': factor_panel['date'].min(),
    '有效因子末日': factor_panel['date'].max(),
    '输出大小MB': factor_path.stat().st_size / 1024 / 1024,
})
display(factor_summary.to_frame('值'))
print('因子文件：', factor_path)
print('下一步：运行', PROJECT_DIR / 'Back_test' / '通用因子回测模板.ipynb')


## 7. 下一步

打开 `Back_test/通用因子回测模板.ipynb`。UPSHADOW 示例已经配置好：

- 因子列：`upshadow_factor`；
- 因子方向：`negative`；
- 因子已经完成 MAD/Z-score：`FACTOR_PREPROCESSED=True`；
- 回测引擎重新从行情 Parquet 构造严格的未来 VWAP 收益，不使用因子文件中的未来信息。